In [ ]:
import pandas as pd

tmp = pd.read_csv('data/dataset-telco.csv', header=None)

df = tmp[0].str.split(',', expand=True)

df.columns = df.iloc[0]

df = df[1:].reset_index(drop=True)

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='ignore')

df.head(10)


In [ ]:
print(df.dtypes)


In [ ]:
print(df['plan_type'].unique())
print(df['device_brand'].unique())
print(df['target_offer'].unique())



In [ ]:
print(df.isnull().sum())

print("Total missing:", df.isnull().sum().sum())


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt



# ======================================================
# 2. ENCODING untuk kolom kategorikal
# ======================================================
le_plan = LabelEncoder()
le_device = LabelEncoder()
le_target = LabelEncoder()

df['plan_type']     = le_plan.fit_transform(df['plan_type'])
df['device_brand']  = le_device.fit_transform(df['device_brand'])
df['target_offer']  = le_target.fit_transform(df['target_offer'])

# ======================================================
# 3. SIAPKAN X dan y
# ======================================================
X = df.drop(["customer_id", "target_offer"], axis=1)  # customer_id tidak berguna
y = df["target_offer"]

# ======================================================
# 4. SPLIT TRAIN / TEST
# ======================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ======================================================
# 5. TRAIN MODEL
# ======================================================
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

# ======================================================
# 6. EVALUASI MODEL
# ======================================================
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Accuracy:", acc)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# confusion matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix (Raw):")
for row in cm:
    print(row)

In [ ]:
import pandas as pd

# Data baru (sama dengan format X)
new_data = {
    "plan_type": ["Prepaid"],
    "device_brand": ["Realme"],
    "avg_data_usage_gb": [9.50],
    "pct_video_usage": [0.804146],
    "avg_call_duration": [20.98],
    "sms_freq": [13],
    "monthly_spend": [70000.0],
    "topup_freq": [4],
    "travel_score": [0.284419],
    "complaint_count": [0]
}

new_df = pd.DataFrame(new_data)

# ====== Encode kolom kategorikal sama seperti training ======
new_df['plan_type']    = le_plan.transform(new_df['plan_type'])
new_df['device_brand'] = le_device.transform(new_df['device_brand'])

# ====== Prediksi kelas ======
pred_class = model.predict(new_df)
pred_class_label = le_target.inverse_transform(pred_class)

print("Predicted Offer:", pred_class_label[0])

# ====== Prediksi probabilitas ======
pred_probs = model.predict_proba(new_df)[0]
prob_df = pd.DataFrame({
    "Offer": le_target.classes_,
    "Probability": pred_probs
}).sort_values(by="Probability", ascending=False)

print("\nProbabilities for each Offer:")
print(prob_df)


In [ ]:
import pandas as pd

# Data baru
new_data = {
    "plan_type": ["Prepaid"],
    "device_brand": ["Oppo"],
    "avg_data_usage_gb": [3.30],
    "pct_video_usage": [0.478873],
    "avg_call_duration": [6.53],
    "sms_freq": [17],
    "monthly_spend": [54000.0],
    "topup_freq": [3],
    "travel_score": [0.372135],
    "complaint_count": [0]
}

new_df = pd.DataFrame(new_data)

# Encode kolom kategorikal sama seperti training
new_df['plan_type']    = le_plan.transform(new_df['plan_type'])
new_df['device_brand'] = le_device.transform(new_df['device_brand'])

# Prediksi kelas
pred_class = model.predict(new_df)
pred_class_label = le_target.inverse_transform(pred_class)
print("Predicted Offer:", pred_class_label[0])

# Prediksi probabilitas
pred_probs = model.predict_proba(new_df)[0]
prob_df = pd.DataFrame({
    "Offer": le_target.classes_,
    "Probability": pred_probs
}).sort_values(by="Probability", ascending=False)

print("\nProbabilities for each Offer:")
print(prob_df)


In [ ]:
# Cek distribusi kelas target
print("Distribusi Kelas target_offer:")
print(df['target_offer'].value_counts())

# Persentase
print("\nPersentase per kelas:")
print(df['target_offer'].value_counts(normalize=True) * 100)

# Visualisasi imbalance
import matplotlib.pyplot as plt

df['target_offer'].value_counts().plot(kind='bar')
plt.title("Distribusi Kelas target_offer")
plt.xlabel("Product / Offer")
plt.ylabel("Jumlah Customer")
plt.show()
